In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import re

from nltk.corpus import stopwords

from nltk.stem import SnowballStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb 

from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import precision_score,recall_score,roc_auc_score

In [2]:
df = pd.read_csv("/Users/narendrafuloria/Desktop/OPS/MLOPS codes for different Libraries/DVC/tweet_emotions.csv")

In [3]:
df.head()

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [4]:
df["sentiment"].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [6]:
df_emotions = df[(df["sentiment"]=="happiness") | (df["sentiment"]=="sadness")]

In [7]:
df_emotions

,tweet_id,sentiment,content
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
6,1956968487,sadness,"I should be sleep, but im not! thinking about ..."
8,1956969035,sadness,@charviray Charlene my love. I miss you
9,1956969172,sadness,@kelcouch I'm sorry at least it's Friday?
...,...,...,...
39986,1753905153,happiness,going to watch boy in the striped pj's hope i ...
39987,1753918809,happiness,"gave the bikes a thorough wash, degrease it an..."
39988,1753918818,happiness,"had SUCH and AMAZING time last night, McFly we..."
39994,1753918900,happiness,Succesfully following Tayla!!


In [8]:
df_emotions["sentiment"].value_counts()

sentiment
happiness    5209
sadness      5165
Name: count, dtype: int64

In [9]:
X = df_emotions.drop(columns= ["sentiment"])

y = df_emotions["sentiment"]

### Natural Language processing steps:
1. convert into lowercase
2. remove the urls and punctuations etc.
3. remove the stop words
4. stemmization and lemmatization
5. follow distribution of words for each sentiment 
6. train a model
7. test


In [11]:
# converting all characters to lowercase

stopwords = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def remove_stopwords(text):
    words = text.split()

    return " ".join(word for word in words if word not in stopwords)
from nltk import pos_tag
from nltk.corpus import wordnet

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    if tag.startswith('V'): return wordnet.VERB
    if tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

def lemmatize_text_pos(text):
    tagged = pos_tag(text.split())
    return ' '.join(lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged)






    








In [12]:
def text_processing(text):

    # removing urls and removing punctuations with space

    # 1. Remove URLs (http://, https://, and www. links)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # 2. Replace every punctuation character with a space
    text = re.sub(r'[^\w\s]', ' ', text)
    # 3. Collapse repeated whitespace left behind
    text = re.sub(r'\s+', ' ', text).strip()

    # converting the text to lowercase

    text = text.lower()

    # removing stopwords

    remove_stopwords(text)

    # applying lemmatization on text

    text = lemmatize_text_pos(text)
    return text



In [13]:
X["content"] = X["content"].apply(text_processing)

In [14]:
X["content"]

1        layin n bed with a headache ughhhh waitin on y...
2                           funeral ceremony gloomy friday
6        i should be sleep but im not think about an ol...
8                    charviray charlene my love i miss you
9                  kelcouch i m sorry at least it s friday
                               ...                        
39986    go to watch boy in the striped pj s hope i don...
39987    give the bike a thorough wash degrease it and ...
39988    have such and amazing time last night mcfly be...
39994                             succesfully follow tayla
39998    niariley wassup beautiful follow me peep out m...
Name: content, Length: 10374, dtype: object

In [15]:
X_train, X_test, y_train,y_test = train_test_split(X,y, test_size= 0.25, random_state=42)

In [16]:
labelencoder = LabelEncoder()

y_train_encoded = labelencoder.fit_transform(y_train)

In [17]:
y_test_encoded = labelencoder.transform(y_test)

In [18]:
X_train = X_train["content"].values

In [19]:
X_test = X_test["content"].values

In [21]:
vectorizer = CountVectorizer()

X_train_bow = vectorizer.fit_transform(X_train)

X_test_bow = vectorizer.transform(X_test)

In [22]:
# Define and train the XGBoost model
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb_model.fit(X_train_bow, y_train_encoded)

# Make predictions
y_pred = xgb_model.predict(X_test_bow)

# Evaluate the model
accuracy = accuracy_score(y_test_encoded, y_pred)
classification_rep = classification_report(y_test_encoded, y_pred)

print("Accuracy:", accuracy)
print("Classification Report:\n", classification_rep)

/Users/narendrafuloria/miniforge3/envs/oil_and_gas/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [19:42:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.7914417887432537
Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.84      0.80      1263
           1       0.83      0.74      0.78      1331

    accuracy                           0.79      2594
   macro avg       0.79      0.79      0.79      2594
weighted avg       0.80      0.79      0.79      2594



In [ ]:
classes = np.unique(y_test_encoded)
print("Classes:", classes)

y_pred_proba = xgb_model.predict_proba(X_test_bow)

if len(classes) == 2:
    # binary: roc_auc_score needs the probability of the positive class only
    auc = roc_auc_score(y_test_encoded, y_pred_proba[:, 1])
else:
    auc = roc_auc_score(y_test_encoded, y_pred_proba, multi_class='ovr', average='weighted')

print("ROC AUC:", auc)

Classes: [0 1]
ROC AUC: 0.8702221167327859
